# Vault AWS Secrets Engine - IAM Role 방식 (IRSA)

이 노트북은 **IAM Role (IRSA / EC2 Instance Profile)**을 통해 Vault가 AWS에 인증하는 방식을 테스트합니다.

HashiCorp 공식 문서 - https://developer.hashicorp.com/vault/docs/secrets/aws

---

## 사전 준비

- EKS 클러스터에서 Vault Pod가 **IRSA**(IAM Roles for Service Accounts)로 IAM Role을 가정(assume)할 수 있어야 함
  - `aws/terraform/modules/vault/main.tf`에서 `vault-role` IAM Role이 IRSA로 설정됨
- IAM Role에 동적 자격증명 발급을 위한 권한 부여
  - `iam:CreateUser`, `iam:AttachUserPolicy`, `iam:CreateAccessKey` 등
  - 또는 STS AssumeRole만 사용할 경우 `sts:AssumeRole` 권한
- AccessKey/SecretKey를 Vault에 **등록하지 않음** → Vault Pod의 IAM Role이 자동으로 사용됨

## 흐름

```
1. Enable  → AWS Secrets Engine 마운트
2. Config  → region만 설정 (자격증명 없음, IRSA 자동 사용)
3. Role    → Vault Role 생성
4. Creds   → 동적 자격증명 발급
5. Lease   → Lease 관리 (갱신/취소)
6. Cleanup → Role 삭제 및 Engine 언마운트
```

## 환경변수 설정

In [ ]:
# direnv 환경변수 로드
direnv allow
eval $(direnv export bash)

# Vault AWS Secrets Engine 마운트 경로
export MOUNT_PATH="my-aws-role"

echo "VAULT_ADDR                    : $VAULT_ADDR"
echo "VAULT_TOKEN                   : ${VAULT_TOKEN:0:10}..."
echo "MOUNT_PATH                    : $MOUNT_PATH"
echo "AWS_REGION                    : $AWS_REGION"
echo "VAULT_ROLE_ARN                : $VAULT_ROLE_ARN"
echo "VAULT_ASSUME_TARGET_ROLE_ARN  : $VAULT_ASSUME_TARGET_ROLE_ARN"

## 1. AWS Secrets Engine 마운트

In [ ]:
# AWS Secrets Engine 마운트 (TTL 설정 포함)
vault secrets enable \
    -path="$MOUNT_PATH" \
    -default-lease-ttl=5m \
    -max-lease-ttl=2h \
    aws

## 2. Root Config 설정 (IAM Role 방식)

In [ ]:
# [기본] region만 설정 - IRSA Role 그대로 사용
vault write "$MOUNT_PATH/config/root" \
  region="$AWS_REGION"

echo "✅ AWS config 등록 완료 (IAM Role 방식)"
echo "   Vault Pod의 IRSA 또는 EC2 Instance Profile이 자동으로 사용됩니다."

## 3. Vault Role 생성

In [ ]:
# Vault Role 생성 - credential_type: assumed_role
vault write "$MOUNT_PATH/roles/irsa-sts-role" \
  credential_type=assumed_role \
  role_arns="$VAULT_ASSUME_TARGET_ROLE_ARN" \
  default_sts_ttl=15m \
  max_sts_ttl=1h

In [ ]:
# Vault Role 생성 - credential_type: iam_user
vault write "$MOUNT_PATH/roles/irsa-iam-user-role" \
  credential_type=iam_user \
  policy_arns="arn:aws:iam::aws:policy/SecretsManagerReadWrite"

In [ ]:
# 생성된 Role 목록 확인
echo "=== Role 목록 ==="
vault list "$MOUNT_PATH/roles"

echo ""
echo "=== irsa-iam-user-role 상세 ==="
vault read "$MOUNT_PATH/roles/irsa-iam-user-role"

echo ""
echo "=== irsa-sts-role 상세 ==="
vault read "$MOUNT_PATH/roles/irsa-sts-role"

## 4. 동적 자격증명 발급 (Generate Credentials)

In [ ]:
# IAM User 자격증명 발급 (iam_user)
echo "=== IAM User 자격증명 발급 ==="
vault read "$MOUNT_PATH/creds/irsa-iam-user-role"

In [ ]:
# STS 임시 자격증명 발급 (assumed_role)
echo "=== STS 임시 자격증명 발급 ==="
vault read "$MOUNT_PATH/sts/irsa-sts-role" ttl=15m

In [ ]:
# JSON 형식으로 발급 후 특정 필드 추출 (IAM User 및 STS 모두 추출)
echo "=== 1. IRSA 기반 IAM User 자격증명 발급 및 추출 ==="
IAM_CREDS=$(vault read -format=json "$MOUNT_PATH/creds/irsa-iam-user-role")
export IAM_LEASE_ID=$(echo $IAM_CREDS | python3 -c "import sys,json; d=json.load(sys.stdin); print(d['lease_id'])")
echo "IAM Lease ID    : $IAM_LEASE_ID"

echo ""
echo "=== 2. IRSA 기반 STS 임시 자격증명 발급 및 추출 ==="
STS_CREDS=$(vault read -format=json "$MOUNT_PATH/sts/irsa-sts-role")
export STS_LEASE_ID=$(echo $STS_CREDS | python3 -c "import sys,json; d=json.load(sys.stdin); print(d['lease_id'])")
DYNA_ACCESS_KEY=$(echo $STS_CREDS | python3 -c "import sys,json; d=json.load(sys.stdin); print(d['data'].get('access_key', ''))")
DYNA_SECRET_KEY=$(echo $STS_CREDS | python3 -c "import sys,json; d=json.load(sys.stdin); print(d['data'].get('secret_key', ''))")
DYNA_SESSION_TOKEN=$(echo $STS_CREDS | python3 -c "import sys,json; d=json.load(sys.stdin); print(d['data'].get('security_token', d['data'].get('session_token', '')))")
echo "STS Lease ID    : $STS_LEASE_ID"
echo "STS AccessKeyId : $DYNA_ACCESS_KEY"

In [ ]:
# 발급된 동적 자격증명을 환경 변수에 할당 (STS 기준)
export AWS_ACCESS_KEY_ID="$DYNA_ACCESS_KEY"
export AWS_SECRET_ACCESS_KEY="$DYNA_SECRET_KEY"
export AWS_SESSION_TOKEN="$DYNA_SESSION_TOKEN"
export AWS_DEFAULT_REGION="$AWS_REGION"

# 1. 현재 인증된 자격증명 정보 확인
echo "=== 1. STS GetCallerIdentity 호출 ==="
aws sts get-caller-identity

In [ ]:
# 2. Secrets Manager에 테스트용 시크릿 생성
echo "=== 2. SecretsManager CreateSecret 호출 ==="
aws secretsmanager create-secret --name test-key-irsa

In [ ]:
# 3. S3 버킷 목록 조회 (권한이 없으므로 오류)
echo "=== 3. S3 ListBuckets 호출 ==="
aws s3 ls

In [ ]:
# 4. 생성한 테스트 시크릿 즉시 삭제
echo "=== 4. SecretsManager DeleteSecret 호출 ==="
aws secretsmanager delete-secret --force-delete-without-recovery --secret-id test-key-irsa

## 5. Lease 관리 (갱신 / 취소)

발급된 동적 자격증명은 **Lease**로 관리됩니다.

In [ ]:
# 활성 Lease 목록 확인
echo "=== IAM User Role Lease 목록 ==="
vault list sys/leases/lookup/${MOUNT_PATH}/creds/irsa-iam-user-role/ || echo "(활성 Lease 없음)"

echo ""
echo "=== STS Role Lease 목록 ==="
vault list sys/leases/lookup/${MOUNT_PATH}/sts/irsa-sts-role/ || echo "(활성 Lease 없음)"

In [ ]:
# lease ID 조회
vault lease lookup ${IAM_LEASE_ID}
echo ""
vault lease lookup ${STS_LEASE_ID}

In [ ]:
# Lease 갱신 (갱신 가능한 IAM User 방식 테스트)
echo "=== IAM User Lease 갱신 (60분 연장 시도) ==="
vault lease renew -increment=3600 "$IAM_LEASE_ID"

In [ ]:
# 특정 Lease 취소 (해당 IAM User/STS Token 즉시 삭제)
echo "=== IAM User Lease 취소 ==="
vault lease revoke "$IAM_LEASE_ID"

In [ ]:
# Role에 속한 모든 Lease 일괄 취소
echo "=== irsa-iam-user-role 의 모든 Lease 취소 ==="
vault lease revoke -prefix "${MOUNT_PATH}/creds/irsa-iam-user-role"

## 6. (정리) AWS Secrets Engine 언마운트

In [ ]:
vault secrets disable "$MOUNT_PATH"

echo ""
vault secrets list